# CIC IDS Sample Inspection

Inspect CIC raw sample values and feature columns to build corrected UI presets for normal, DDoS, and portscan cases.

In [2]:
import pandas as pd
import pickle
from pathlib import Path

base = Path(r'c:/Users/OBITO/Desktop/IDS with DL/IDS with DL - CIC2017')
feature_columns = pickle.load(open(base / 'models' / 'feature_columns.pkl', 'rb'))
print('Feature count:', len(feature_columns))
print('First 20 features:', feature_columns[:20])

mon = pd.read_csv(base / 'data' / 'raw' / 'Monday-WorkingHours.pcap_ISCX.csv')
print('Monday columns:', list(mon.columns[:20]))
label_col = [c for c in mon.columns if c.strip().lower() == 'label']
if not label_col:
    label_col = [c for c in mon.columns if 'label' in c.strip().lower()]
label_col = label_col[0]
print('Detected label column:', label_col)

benign = mon[mon[label_col].astype(str).str.strip() == 'BENIGN'].iloc[0]
ddos = pd.read_csv(base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
if label_col not in ddos.columns:
    ddos = ddos.rename(columns={ [c for c in ddos.columns if c.strip().lower() == 'label'][0]: label_col })
ddos = ddos[ddos[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]
port = pd.read_csv(base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
if label_col not in port.columns:
    port = port.rename(columns={ [c for c in port.columns if c.strip().lower() == 'label'][0]: label_col })
port = port[port[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]

print('BENIGN sample label:', benign[label_col])
print('DDoS sample label:', ddos[label_col])
print('Portscan sample label:', port[label_col])

def sample_dict(row, cols):
    return {col: row[col] if col in row else 0 for col in cols}

normal_sample = sample_dict(benign, feature_columns)
ddos_sample = sample_dict(ddos, feature_columns)
portscan_sample = sample_dict(port, feature_columns)

print('Normal sample size:', len(normal_sample))
print('DDoS sample size:', len(ddos_sample))
print('Portscan sample size:', len(portscan_sample))

for name, sample in [('normal', normal_sample), ('dos', ddos_sample), ('portscan', portscan_sample)]:
    print('\n===', name, 'sample preview ===')
    for k in list(sample)[:20]:
        print(k, ':', sample[k])


Feature count: 78
First 20 features: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min']
Monday columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min']
Detected label column:  Label
BENI

In [5]:
# Inspect non-zero CIC sample features for attack presets
for name, row in [('ddos', ddos), ('portscan', port)]:
    print(f"\n=== {name.upper()} non-zero features ===")
    nonzeros = {k: v for k, v in row.items() if pd.notna(v) and v != 0 and k in feature_columns}
    for k, v in list(nonzeros.items())[:40]:
        print(k, ':', v)
    print(f"Total non-zero features: {len(nonzeros)}")



=== DDOS non-zero features ===
Total Length of Fwd Packets : 26
Bwd Packet Length Max : 5840
Flow Bytes/s : 8991.398927
Fwd IAT Total : 747
Bwd IAT Total : 1293746
Fwd Packets/s : 2.318765304
Subflow Fwd Packets : 3
Init_Win_bytes_forward : 8192
Total non-zero features: 8

=== PORTSCAN non-zero features ===
Total Length of Fwd Packets : 703
Bwd Packet Length Max : 1050
Flow Bytes/s : 421.6242032
Fwd IAT Total : 55401
Bwd IAT Total : 5020928
Fwd Packets/s : 1.194967038
Subflow Fwd Packets : 6
Init_Win_bytes_forward : 29200
Total non-zero features: 8


In [7]:
# Check the exact columns available in CIC raw attack files
for name, path in [('ddos', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
                   ('portscan', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')]:
    df = pd.read_csv(path)
    print(f"\n=== {name.upper()} columns ({len(df.columns)}) ===")
    print(df.columns.tolist()[:40])
    print('...')
    print(df.columns.tolist()[-10:])



=== DDOS columns (79) ===
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length']
...
[' min_seg_size_forward', 'Active Mean', ' Active Std', ' Active Max', ' Active Min', 'Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min', ' Label'

In [8]:
# Re-extract CIC sample presets with stripped column names for raw files
for name, path in [('ddos', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
                   ('portscan', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')]:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    label_col = [c for c in df.columns if c.strip().lower() == 'label'][0]
    row = df[df[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]
    sample = {col: row[col] if col in row else 0 for col in feature_columns}
    nonzeros = {k: v for k, v in sample.items() if pd.notna(v) and v != 0}
    print(f"\n=== {name.upper()} sample non-zero features ({len(nonzeros)}) ===")
    for k, v in list(nonzeros.items())[:40]:
        print(k, ':', v)
    print(f"Total sample keys: {len(sample)}")
    print("Example keys with zero values:", [k for k, v in sample.items() if v == 0][:10])



=== DDOS sample non-zero features (50) ===
Destination Port : 80
Flow Duration : 1293792
Total Fwd Packets : 3
Total Backward Packets : 7
Total Length of Fwd Packets : 26
Total Length of Bwd Packets : 11607
Fwd Packet Length Max : 20
Fwd Packet Length Mean : 8.666666667
Fwd Packet Length Std : 10.26320288
Bwd Packet Length Max : 5840
Bwd Packet Length Mean : 1658.142857
Bwd Packet Length Std : 2137.29708
Flow Bytes/s : 8991.398927
Flow Packets/s : 7.72921768
Flow IAT Mean : 143754.6667
Flow IAT Std : 430865.8067
Flow IAT Max : 1292730
Flow IAT Min : 2
Fwd IAT Total : 747
Fwd IAT Mean : 373.5
Fwd IAT Std : 523.9661249
Fwd IAT Max : 744
Fwd IAT Min : 3
Bwd IAT Total : 1293746
Bwd IAT Mean : 215624.3333
Bwd IAT Std : 527671.9348
Bwd IAT Max : 1292730
Bwd IAT Min : 2
Fwd Header Length : 72
Bwd Header Length : 152
Fwd Packets/s : 2.318765304
Bwd Packets/s : 5.410452376
Max Packet Length : 5840
Packet Length Mean : 1057.545455
Packet Length Std : 1853.437529
Packet Length Variance : 3435230

In [9]:
for name, path in [('ddos', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
                   ('portscan', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')]:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    label_col = [c for c in df.columns if c.lower() == 'label'][0]
    row = df[df[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]
    sample = {col: row[col] if col in row else 0 for col in feature_columns}
    print(f"\n=== {name.upper()} sample non-zero features ===")
    nonzeros = [(k, v) for k, v in sample.items() if pd.notna(v) and v != 0]
    for k, v in nonzeros[:20]:
        print(f"{k}: {v}")
    print(f"... total nonzeros: {len(nonzeros)}")
    # print only a handful of zero fields to confirm mapping
    zeros = [k for k, v in sample.items() if v == 0][:20]
    print('Zero sample keys snippet:', zeros)



=== DDOS sample non-zero features ===
Destination Port: 80
Flow Duration: 1293792
Total Fwd Packets: 3
Total Backward Packets: 7
Total Length of Fwd Packets: 26
Total Length of Bwd Packets: 11607
Fwd Packet Length Max: 20
Fwd Packet Length Mean: 8.666666667
Fwd Packet Length Std: 10.26320288
Bwd Packet Length Max: 5840
Bwd Packet Length Mean: 1658.142857
Bwd Packet Length Std: 2137.29708
Flow Bytes/s: 8991.398927
Flow Packets/s: 7.72921768
Flow IAT Mean: 143754.6667
Flow IAT Std: 430865.8067
Flow IAT Max: 1292730
Flow IAT Min: 2
Fwd IAT Total: 747
Fwd IAT Mean: 373.5
... total nonzeros: 50
Zero sample keys snippet: ['Fwd Packet Length Min', 'Bwd Packet Length Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Min Packet Length', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk

In [11]:
# Generate JS object text for corrected CIC sample presets
def js_object(code_name, sample):
    lines = [f"    {code_name}: {{"]
    for k in feature_columns:
        v = sample[k]
        if isinstance(v, str):
            v_text = "'" + v.replace("'", "\\'") + "'"
        elif isinstance(v, float):
            v_text = repr(v)
        else:
            v_text = str(int(v)) if pd.api.types.is_integer_dtype(type(v)) else repr(v)
        lines.append(f"      '{k}': {v_text},")
    lines[-1] = lines[-1].rstrip(',')
    lines.append('    },')
    return '\n'.join(lines)

for name, path in [('dos', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
                   ('portscan', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')]:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    label_col = [c for c in df.columns if c.lower() == 'label'][0]
    row = df[df[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]
    sample = {col: row[col] if col in row else 0 for col in feature_columns}
    print(js_object(name, sample))


    dos: {
      'Destination Port': 80,
      'Flow Duration': 1293792,
      'Total Fwd Packets': 3,
      'Total Backward Packets': 7,
      'Total Length of Fwd Packets': 26,
      'Total Length of Bwd Packets': 11607,
      'Fwd Packet Length Max': 20,
      'Fwd Packet Length Min': 0,
      'Fwd Packet Length Mean': 8.666666667,
      'Fwd Packet Length Std': 10.26320288,
      'Bwd Packet Length Max': 5840,
      'Bwd Packet Length Min': 0,
      'Bwd Packet Length Mean': 1658.142857,
      'Bwd Packet Length Std': 2137.29708,
      'Flow Bytes/s': 8991.398927,
      'Flow Packets/s': 7.72921768,
      'Flow IAT Mean': 143754.6667,
      'Flow IAT Std': 430865.8067,
      'Flow IAT Max': 1292730,
      'Flow IAT Min': 2,
      'Fwd IAT Total': 747,
      'Fwd IAT Mean': 373.5,
      'Fwd IAT Std': 523.9661249,
      'Fwd IAT Max': 744,
      'Fwd IAT Min': 3,
      'Bwd IAT Total': 1293746,
      'Bwd IAT Mean': 215624.3333,
      'Bwd IAT Std': 527671.9348,
      'Bwd IAT Max':

In [12]:
# Save JS object literals for corrected CIC presets into a temp file
with open(base.parent / 'cic_cic_js_presets.txt', 'w', encoding='utf-8') as out:
    for name, path in [('dos', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
                       ('portscan', base / 'data' / 'raw' / 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')]:
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        label_col = [c for c in df.columns if c.lower() == 'label'][0]
        row = df[df[label_col].astype(str).str.strip() != 'BENIGN'].iloc[0]
        sample = {col: row[col] if col in row else 0 for col in feature_columns}
        out.write(js_object(name, sample) + '\n')
print('Saved corrected JS preset text to cic_cic_js_presets.txt')


Saved corrected JS preset text to cic_cic_js_presets.txt


In [4]:
import json

def normalized(obj):
    if hasattr(obj, 'item'):
        return obj.item()
    return obj

preset_data = {
    'normal': {k: normalized(v) for k, v in normal_sample.items()},
    'dos': {k: normalized(v) for k, v in ddos_sample.items()},
    'portscan': {k: normalized(v) for k, v in portscan_sample.items()},
}
with open(r'c:/Users/OBITO/Desktop/IDS with DL/cic_cic_presets.json', 'w', encoding='utf-8') as f:
    json.dump(preset_data, f, indent=2)
print('Saved presets to cic_cic_presets.json')

Saved presets to cic_cic_presets.json
